# NeuroScan Phase 2 — Track 1 (Kainat)

**Baseline CNN + the data-leakage experiment.**

Phase 1 found that the raw dataset contains 187 exact duplicates, 90 near-duplicates and
**161 training images that also appear in the test set**. The original Phase 2 notebook
trained straight off the raw directories, so it never benefited from any of that cleaning.

This notebook does two things:

1. **A from-scratch CNN baseline** on the clean pipeline — the reference point that says how
   much transfer learning is actually buying us (Laiba's and Mania's tracks).
2. **The leakage experiment** — train the *same* EfficientNetB0, with the *same*
   hyperparameters, once on the clean data and once on the old leaky pipeline, and measure
   the gap. That number is what Phase 1 was for.

> **Run this on a GPU.** Colab: *Runtime → Change runtime type → T4 GPU*. On CPU the
> EfficientNet runs take hours.

Everything shared lives in `neuroscan/` — see `PHASE2.md` for the contract.

## 1. Setup

In [1]:
# Colab: clone the repo. Local: this cell is a no-op.
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO = "NeuroScan"

if IN_COLAB and not os.path.isdir(REPO):
    # Use your own fork/remote here if you are not pushing to this one.
    subprocess.run(["git", "clone", "https://github.com/laibaamjad50/NeuroScan.git"], check=True)
if IN_COLAB:
    os.chdir(REPO)

print("working directory:", os.getcwd())
assert os.path.isdir("neuroscan"), "run this from the repository root"

working directory: /content/NeuroScan


In [2]:
!pip install -q -r requirements-phase2.txt

In [3]:
# processed_data/ is gitignored, so rebuild it from the manifest (a few minutes).
# Reproduces the Phase 1 output exactly; skips anything already on disk.
!python scripts/build_processed_data.py

manifest rows     : 6,762
already on disk   : 0
to build          : 6,762
Preprocessing: 100% 6762/6762 [00:57<00:00, 117.21it/s]

built 6,762 | skipped 0 | failed 0
processed_data/ now holds 6,762 of 6,762 expected images


In [4]:
import json, os, sys
import numpy as np
import tensorflow as tf
from tensorflow.keras import layers, models, callbacks, applications

sys.path.insert(0, os.getcwd())
from neuroscan import data, evaluate

SEED = 42
tf.keras.utils.set_random_seed(SEED)

EPOCHS_BASELINE = 25
EPOCHS_PHASE_A  = 15     # matches the original Phase 2 notebook
EPOCHS_PHASE_B  = 15

os.makedirs("models", exist_ok=True)
os.makedirs("results", exist_ok=True)

print("TensorFlow", tf.__version__)
print("GPU:", tf.config.list_physical_devices("GPU") or "NONE - this will be slow")

report = data.verify(full=True)
print("\nPhase 1 outputs:", "OK" if report["ok"] else "PROBLEM")
for p in report["problems"]:
    print("  -", p)
print("counts:", report["counts"])
CLASSES = report["classes"]
print("classes:", CLASSES)

TensorFlow 2.21.0
GPU: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

Phase 1 outputs: OK
counts: {'train': 4156, 'val': 1039, 'test': 1567}
classes: ['glioma', 'meningioma', 'notumor', 'pituitary']


## 2. Baseline — a small CNN trained from scratch

No pretrained weights, single greyscale channel, inputs scaled to [0, 1]. This is the
"what do we get without transfer learning?" number. If a 21-million-parameter pretrained
backbone can't beat this by a clear margin, that's worth knowing.

In [5]:
def build_baseline_cnn(input_shape, n_classes=4):
    """Four conv blocks -> global average pool -> dense head."""
    return models.Sequential([
        layers.Input(shape=input_shape),
        *[l for f in (32, 64, 128, 256) for l in (
            layers.Conv2D(f, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.Conv2D(f, 3, padding="same", use_bias=False),
            layers.BatchNormalization(),
            layers.ReLU(),
            layers.MaxPooling2D(),
        )],
        layers.GlobalAveragePooling2D(),
        layers.Dropout(0.4),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.3),
        layers.Dense(n_classes, activation="softmax"),
    ], name="baseline_cnn")


# 1 channel + [0,1] scaling: appropriate for a network trained from scratch.
cfg_base = data.DataConfig(preprocess="rescale", channels=1, batch_size=32, seed=SEED)
ds_base = data.make_datasets(cfg_base)
info = ds_base["info"]
print(info["counts"], "| input", info["input_shape"])

baseline = build_baseline_cnn(info["input_shape"], len(CLASSES))
baseline.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])
baseline.summary()

{'train': 4156, 'val': 1039, 'test': 1567} | input (224, 224, 1)


Model: "baseline_cnn"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 224, 224, 32)   │           288 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization             │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu (ReLU)                    │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 224, 224, 32)   │         9,216 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_1           │ (None, 224, 224, 32)   │           128 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_1 (ReLU)                  │ (None, 224, 224, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d (MaxPooling2D)    │ (None, 112, 112, 32)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_2 (Conv2D)               │ (None, 112, 112, 64)   │        18,432 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_2           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_2 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_3 (Conv2D)               │ (None, 112, 112, 64)   │        36,864 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 112, 112, 64)   │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_3 (ReLU)                  │ (None, 112, 112, 64)   │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_1 (MaxPooling2D)  │ (None, 56, 56, 64)     │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_4 (Conv2D)               │ (None, 56, 56, 128)    │        73,728 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_4 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_5 (Conv2D)               │ (None, 56, 56, 128)    │       147,456 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 56, 56, 128)    │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ re_lu_5 (ReLU)                  │ (None, 56, 56, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling2d_2 (MaxPooling2D)  │ (None, 28, 28, 128)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_6 (Conv2D)               │ (None, 28, 28, 256)    │       294,91

 Total params: 1,241,380 (4.74 MB)

 Trainable params: 1,239,460 (4.73 MB)

 Non-trainable params: 1,920 (7.50 KB)

In [6]:
cb = [
    callbacks.ModelCheckpoint("models/baseline_cnn.keras", monitor="val_accuracy",
                              save_best_only=True, verbose=1),
    callbacks.EarlyStopping(monitor="val_loss", patience=6,
                            restore_best_weights=True, verbose=1),
    callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=3, verbose=1),
]

hist_base = baseline.fit(
    ds_base["train"],
    validation_data=ds_base["val"],
    epochs=EPOCHS_BASELINE,
    class_weight=info["class_weights"],
    callbacks=cb,
)

Epoch 1/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 443ms/step - accuracy: 0.5782 - loss: 0.9918
Epoch 1: val_accuracy improved from None to 0.26853, saving model to models/baseline_cnn.keras

Epoch 1: finished saving model to models/baseline_cnn.keras
130/130 ━━━━━━━━━━━━━━━━━━━━ 111s 529ms/step - accuracy: 0.6410 - loss: 0.8615 - val_accuracy: 0.2685 - val_loss: 3.3003 - learning_rate: 0.0010
Epoch 2/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 210ms/step - accuracy: 0.7136 - loss: 0.7137
Epoch 2: val_accuracy did not improve from 0.26853
130/130 ━━━━━━━━━━━━━━━━━━━━ 32s 222ms/step - accuracy: 0.7324 - loss: 0.6808 - val_accuracy: 0.2685 - val_loss: 6.0604 - learning_rate: 0.0010
Epoch 3/25
130/130 ━━━━━━━━━━━━━━━━━━━━ 0s 209ms/step - accuracy: 0.7574 - loss: 0.6175
Epoch 3: val_accuracy improved from 0.26853 to 0.28200, saving model to models/baseline_cnn.keras

Epoch 3: finished saving model to models/baseline_cnn.keras
130/130 ━━━━━━━━━━━━━━━━━━━━ 33s 232ms/step - accuracy: 0.7618 - loss: 0.6053 - v

In [7]:
res_base = evaluate.evaluate_model(baseline, ds_base["test"], CLASSES,
                                   name="baseline_cnn", outdir="results")
evaluate.plot_history(hist_base, "results/baseline_cnn_history.png", "baseline CNN")

print(open("results/baseline_cnn_report.txt").read())

baseline_cnn
samples            1567
accuracy           0.8060
balanced accuracy  0.8052
macro F1           0.7976
macro precision    0.8304
macro recall       0.8052

class             prec     rec      f1      n
glioma          0.9801  0.6423  0.7760    383
meningioma      0.7767  0.5884  0.6695    396
notumor         0.6867  1.0000  0.8143    388
pituitary       0.8780  0.9900  0.9307    400

tumours called 'no tumour': 177 (15.01%)



## 3. The leakage experiment

The design: **one model, one set of hyperparameters, two datasets.**

| run | training data | test data |
|---|---|---|
| `effnet_clean` | Phase 1 clean split (4,156) | clean test (1,567) |
| `effnet_leaky` | raw `dataset/Training` with a random 20% val split | raw `dataset/Testing` (1,600) |

The leaky run reproduces exactly what the original Phase 2 notebook did. Its training set
still contains the 161 images that also sit in its test set, so it can score well by
recognising images it has memorised.

We also evaluate the **leaky-trained model on the clean test set**. That separates two
different effects: how much came from training on contaminated data, versus how much came
from the test set itself containing duplicates.

Hyperparameters below are copied from the original notebook so the comparison is about the
data and nothing else.

In [8]:
def build_effnet(input_shape, n_classes=4):
    """EfficientNetB0 + the same head as the original Phase 2 notebook."""
    base = applications.EfficientNetB0(
        input_shape=input_shape, include_top=False, weights="imagenet")
    base.trainable = False
    model = models.Sequential([
        base,
        layers.GlobalAveragePooling2D(),
        layers.BatchNormalization(),
        layers.Dense(256, activation="relu"),
        layers.Dropout(0.4),
        layers.Dense(n_classes, activation="softmax"),
    ], name="effnet")
    return model, base


def train_effnet(ds, tag, class_weight=None):
    """Two-phase training: frozen base, then fine-tune at a low learning rate."""
    tf.keras.utils.set_random_seed(SEED)
    model, base = build_effnet(ds["info"]["input_shape"], len(CLASSES))
    model.compile(optimizer="adam", loss="categorical_crossentropy", metrics=["accuracy"])

    cb = [
        callbacks.ModelCheckpoint(f"models/{tag}.keras", monitor="val_accuracy",
                                  save_best_only=True, verbose=0),
        callbacks.EarlyStopping(monitor="val_loss", patience=5,
                                restore_best_weights=True, verbose=1),
    ]

    print(f"\n=== {tag} | phase A (base frozen) ===")
    h_a = model.fit(ds["train"], validation_data=ds["val"], epochs=EPOCHS_PHASE_A,
                    class_weight=class_weight, callbacks=cb)

    print(f"\n=== {tag} | phase B (fine-tuning) ===")
    base.trainable = True
    model.compile(optimizer=tf.keras.optimizers.Adam(1e-5),
                  loss="categorical_crossentropy", metrics=["accuracy"])
    h_b = model.fit(ds["train"], validation_data=ds["val"], epochs=EPOCHS_PHASE_B,
                    class_weight=class_weight, callbacks=cb)

    evaluate.plot_history([h_a, h_b], f"results/{tag}_history.png", tag)
    return model, [h_a, h_b]

In [9]:
# --- Run 1: the clean pipeline -------------------------------------------
cfg_eff = data.DataConfig(preprocess="efficientnet", channels=3, batch_size=32, seed=SEED)
ds_clean = data.make_datasets(cfg_eff)

eff_clean, hist_clean = train_effnet(ds_clean, "effnet_clean",
                                     class_weight=ds_clean["info"]["class_weights"])
res_clean = evaluate.evaluate_model(eff_clean, ds_clean["test"], CLASSES,
                                    name="effnet_clean", outdir="results")
print(f"clean  -> accuracy {res_clean['accuracy']:.4f} | macro-F1 {res_clean['macro_f1']:.4f}")

16705208/16705208 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step

=== effnet_clean | phase A (base frozen) ===
Epoch 1/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 136s 641ms/step - accuracy: 0.7736 - loss: 0.6761 - val_accuracy: 0.8807 - val_loss: 0.3909
Epoch 2/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 41s 161ms/step - accuracy: 0.8547 - loss: 0.4036 - val_accuracy: 0.8672 - val_loss: 0.3479
Epoch 3/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 41s 161ms/step - accuracy: 0.8744 - loss: 0.3324 - val_accuracy: 0.8710 - val_loss: 0.3684
Epoch 4/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 24s 162ms/step - accuracy: 0.8898 - loss: 0.3048 - val_accuracy: 0.8807 - val_loss: 0.2978
Epoch 5/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 24s 161ms/step - accuracy: 0.8833 - loss: 0.3009 - val_accuracy: 0.9115 - val_loss: 0.2652
Epoch 6/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 24s 151ms/step - accuracy: 0.9098 - loss: 0.2425 - val_accuracy: 0.8912 - val_loss: 0.2598
Epoch 7/15
130/130 ━━━━━━━━━━━━━━━━━━━━ 41s 161ms/step - accuracy: 0.9090 - loss: 0.2379 - val_accuracy: 0.9018 - va

In [10]:
# --- Run 2: the old leaky pipeline ---------------------------------------
# Same model, same hyperparameters, contaminated data. No class weights here,
# because the original notebook did not use any.
ds_leaky = data.make_leaky_datasets(cfg_eff)
print("leaky pipeline:", ds_leaky["info"]["counts"])

eff_leaky, hist_leaky = train_effnet(ds_leaky, "effnet_leaky", class_weight=None)
res_leaky = evaluate.evaluate_model(eff_leaky, ds_leaky["test"], CLASSES,
                                    name="effnet_leaky", outdir="results")
print(f"leaky  -> accuracy {res_leaky['accuracy']:.4f} | macro-F1 {res_leaky['macro_f1']:.4f}")

Found 5600 files belonging to 4 classes.
Using 4480 files for training.
Found 5600 files belonging to 4 classes.
Using 1120 files for validation.
Found 1600 files belonging to 4 classes.
leaky pipeline: {'train': 4480, 'val': 1120, 'test': 1600}

=== effnet_leaky | phase A (base frozen) ===
Epoch 1/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 53s 224ms/step - accuracy: 0.7801 - loss: 0.6865 - val_accuracy: 0.9777 - val_loss: 0.1813
Epoch 2/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 64s 172ms/step - accuracy: 0.8420 - loss: 0.4387 - val_accuracy: 0.9634 - val_loss: 0.1203
Epoch 3/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 44s 192ms/step - accuracy: 0.8585 - loss: 0.3849 - val_accuracy: 0.9893 - val_loss: 0.0402
Epoch 4/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 26s 184ms/step - accuracy: 0.8879 - loss: 0.3075 - val_accuracy: 0.9937 - val_loss: 0.0214
Epoch 5/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 25s 179ms/step - accuracy: 0.9013 - loss: 0.2818 - val_accuracy: 0.9911 - val_loss: 0.0294
Epoch 6/15
140/140 ━━━━━━━━━━━━━━━━━━━━ 25s 180ms/ste

In [11]:
# --- Run 2b: the leaky-trained model, scored on the CLEAN test set --------
# Isolates "trained on contaminated data" from "tested on a contaminated test set".
res_cross = evaluate.evaluate_model(eff_leaky, ds_clean["test"], CLASSES,
                                    name="effnet_leaky_on_clean_test", outdir="results")
print(f"leaky model / clean test -> accuracy {res_cross['accuracy']:.4f} "
      f"| macro-F1 {res_cross['macro_f1']:.4f}")

leaky model / clean test -> accuracy 0.7498 | macro-F1 0.7470


## 4. Results

In [12]:
gap_acc = res_leaky["accuracy"] - res_clean["accuracy"]
gap_f1  = res_leaky["macro_f1"] - res_clean["macro_f1"]

print(f"{'run':<34}{'accuracy':>10}{'macro-F1':>10}")
for label, r in (("clean pipeline (honest)", res_clean),
                 ("leaky pipeline (old notebook)", res_leaky),
                 ("leaky model on clean test", res_cross)):
    print(f"{label:<34}{r['accuracy']:>10.4f}{r['macro_f1']:>10.4f}")

print(f"\nInflation from leakage: {gap_acc:+.4f} accuracy, {gap_f1:+.4f} macro-F1")
print(f"i.e. the uncleaned pipeline reports about {abs(gap_acc) * 100:.2f} "
      f"percentage points more accuracy than it has earned.")

summary = {
    "clean": {k: res_clean[k] for k in ("accuracy", "macro_f1")},
    "leaky": {k: res_leaky[k] for k in ("accuracy", "macro_f1")},
    "leaky_on_clean_test": {k: res_cross[k] for k in ("accuracy", "macro_f1")},
    "inflation_accuracy": gap_acc,
    "inflation_macro_f1": gap_f1,
}
with open("results/leakage_experiment.json", "w") as fh:
    json.dump(summary, fh, indent=2)

evaluate.comparison_table("results")

run                                 accuracy  macro-F1
clean pipeline (honest)               0.6943    0.6850
leaky pipeline (old notebook)         0.7469    0.7440
leaky model on clean test             0.7498    0.7470

Inflation from leakage: +0.0526 accuracy, +0.0590 macro-F1
i.e. the uncleaned pipeline reports about 5.26 percentage points more accuracy than it has earned.


,model,n,accuracy,balanced_acc,macro_f1,macro_recall,macro_auc,tumour_missed
0,baseline_cnn,1567,0.8060,0.8052,0.7976,0.8052,0.9336,0.1501
1,effnet_leaky_on_clean_test,1567,0.7498,0.7511,0.7470,0.7511,0.9363,0.0933
2,effnet_leaky,1600,0.7469,0.7469,0.7440,0.7469,0.9351,0.0917
3,effnet_clean,1567,0.6943,0.6961,0.6850,0.6961,0.9121,0.1323


## 5. Those numbers are wrong — what happened

**Do not quote the table above.** Both EfficientNet runs were damaged by the fine-tuning
phase, and the figures in Section 4 come from the damaged models.

The training logs show it clearly. `effnet_clean` reached **val_accuracy 0.9355** at epoch 12
of phase A. Phase B then collapsed it to 0.7584 in a single epoch and it never recovered:

```
phase A  epoch 12   val_accuracy 0.9355   <- good model
phase B  epoch 1    val_accuracy 0.7584   <- collapsed
         epoch 5    early stopping
         "Restoring model weights from the end of the best epoch: 1"
```

Two causes, both in `train_effnet`:

1. **Unfrozen BatchNorm.** `base.trainable = True` makes every BatchNormalization layer in
   EfficientNetB0 trainable, so it recomputes its moving statistics on batches of 32 and the
   pretrained features degrade. BN layers must stay frozen while fine-tuning:

   ```python
   base.trainable = True
   for layer in base.layers:
       if isinstance(layer, layers.BatchNormalization):
           layer.trainable = False
   ```

2. **`restore_best_weights` is per-`fit()`.** EarlyStopping resets its state on each call, so
   after phase B it restored the best *phase B* epoch — worse than anything in phase A. The
   original Phase 2 notebook avoided this by reloading the checkpoint before scoring
   (`load_model('best_model.keras')`); that step was missing here.

`ModelCheckpoint` does *not* reset between calls, so `models/*.keras` still held the phase-A
weights. The cells below reload those checkpoints and redo the evaluation — no retraining
needed.

In [13]:
from google.colab import drive
import os, shutil
drive.mount('/content/drive')

dest = '/content/drive/MyDrive/NeuroScan_Phase2'
os.makedirs(dest, exist_ok=True)
for d in ('models', 'results'):
    if os.path.isdir(d):
        shutil.copytree(d, f'{dest}/{d}', dirs_exist_ok=True)
        print(d, '->', len(os.listdir(f'{dest}/{d}')), 'files')

Mounted at /content/drive
models -> 3 files
results -> 18 files


In [14]:
import tensorflow as tf
from neuroscan import evaluate

best_clean = tf.keras.models.load_model("models/effnet_clean.keras")
best_leaky = tf.keras.models.load_model("models/effnet_leaky.keras")

r_clean = evaluate.evaluate_model(best_clean, ds_clean["test"], CLASSES,
                                  name="effnet_clean_best", outdir="results")
r_leaky = evaluate.evaluate_model(best_leaky, ds_leaky["test"], CLASSES,
                                  name="effnet_leaky_best", outdir="results")
r_cross = evaluate.evaluate_model(best_leaky, ds_clean["test"], CLASSES,
                                  name="effnet_leaky_best_on_clean_test", outdir="results")

print(f"{'run':<34}{'accuracy':>10}{'macro-F1':>10}")
for lbl, r in (("clean (phase-A checkpoint)", r_clean),
               ("leaky (phase-A checkpoint)", r_leaky),
               ("leaky model, clean test", r_cross)):
    print(f"{lbl:<34}{r['accuracy']:>10.4f}{r['macro_f1']:>10.4f}")
print(f"\nvs the phase-B models you evaluated: 0.6943 clean / 0.7469 leaky")

run                                 accuracy  macro-F1
clean (phase-A checkpoint)            0.8551    0.8500
leaky (phase-A checkpoint)            0.8850    0.8816
leaky model, clean test               0.8858    0.8823

vs the phase-B models you evaluated: 0.6943 clean / 0.7469 leaky


In [15]:
import os, numpy as np, pandas as pd
from PIL import Image
from neuroscan import data, evaluate

def dhash(path):                      # same definition Phase 1 used
    g = Image.open(path).convert('L')
    s = np.asarray(g.resize((17, 16), Image.LANCZOS), dtype=np.int16)
    return np.packbits((s[:, 1:] > s[:, :-1]).ravel()).tobytes().hex()

log = pd.read_csv('data/cleaning_log.csv')
meta = pd.read_csv('data/cleaned_metadata.csv')
tmeta = meta[meta.split == 'test']
by_dhash = dict(zip(tmeta.dhash, tmeta.path))

leaked = set()
for p in log[log.drop_reason.str.contains('leakage')].path:
    p = p.replace('\\', '/')
    if os.path.exists(p):
        h = dhash(p)
        if h in by_dhash:
            leaked.add(by_dhash[h])

rows = data.load_manifest().query("final_split == 'test'")
mask = np.array([o in leaked for o in rows.original_path])
y_true = rows.label_idx.astype(int).values
CLS = data.class_names()
print(f"test {len(mask)} | leaked {mask.sum()} | non-leaked {(~mask).sum()}\n")

print(f"{'model':<16}{'subset':<14}{'accuracy':>10}{'macro-F1':>10}")
for name, m in (("clean-trained", best_clean), ("leaky-trained", best_leaky)):
    yp = m.predict(ds_clean["test"], verbose=0).argmax(1)
    for lbl, sel in (("leaked", mask), ("non-leaked", ~mask)):
        r = evaluate.compute_metrics(y_true[sel], yp[sel], CLS)
        print(f"{name:<16}{lbl:<14}{r['accuracy']:>10.4f}{r['macro_f1']:>10.4f}")

test 1567 | leaked 155 | non-leaked 1412

model           subset          accuracy  macro-F1


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


clean-trained   leaked            0.5419    0.3859
clean-trained   non-leaked        0.8895    0.8847
leaky-trained   leaked            0.8258    0.5494
leaky-trained   non-leaked        0.8924    0.8855


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:2524: UserWarning: y_pred contains classes not in y_true
  warnings.warn("y_pred contains classes not in y_true")


In [16]:
for d in ('models', 'results'):
    shutil.copytree(d, f'{dest}/{d}', dirs_exist_ok=True)
    print(d, '->', len(os.listdir(f'{dest}/{d}')), 'files')

models -> 3 files
results -> 27 files


## 6. Conclusions

### The fine-tuning bug cost 16 points

| model | as evaluated in Section 4 | from the phase-A checkpoint |
|---|---|---|
| `effnet_clean` | 0.6943 | **0.8551** |
| `effnet_leaky` | 0.7469 | **0.8850** |

The models were never bad; the evaluation was reading the wrong weights.

### Leakage inflates the reported score by 3.1 points

| run | accuracy | macro-F1 |
|---|---|---|
| clean-trained, clean test | 0.8551 | 0.8500 |
| leaky-trained, clean test | 0.8858 | 0.8823 |
| **difference** | **+3.07** | **+3.23** |

The leaky pipeline also trained on 324 more images and used no class weights, so the gap
alone doesn't prove leakage caused it. Splitting the test set settles it.

### All of the gap comes from the 9.9% of test images that leaked

155 of the 1,567 clean test images are near-duplicates of training images the leaky model
saw. (Phase 1 removed the *training*-side copies; the test-side twins remain, which is why
swapping test sets changed nothing.)

| model | leaked (155) | non-leaked (1,412) |
|---|---|---|
| clean-trained | 0.5419 | 0.8895 |
| leaky-trained | **0.8258** | 0.8924 |
| difference | **+28.39** | +0.29 |

On images neither model memorised the two are indistinguishable — 0.3 points — so the extra
data and the class weights bought essentially nothing. The entire advantage sits in the
leaked slice, and the decomposition reproduces the headline gap exactly:

```
0.099 x (+0.2839)  +  0.901 x (+0.0029)  =  +0.0307
observed gap                                +0.0307
```

### Two caveats

**Use accuracy, not macro-F1, on the leaked subset.** Those 155 images contain no pituitary
at all and only 4 glioma, so macro-F1 averages over classes with no support and the
0.3859 / 0.5494 values are not meaningful. (This is what the sklearn warning reports.)

**The leakage is concentrated, not spread evenly:**

| class | test images | leaked | share |
|---|---|---|---|
| meningioma | 396 | 98 | 24.7% |
| notumor | 388 | 53 | 13.7% |
| glioma | 383 | 4 | 1.0% |
| pituitary | 400 | 0 | 0% |

Nearly a quarter of the meningioma test set was compromised. The dataset contains augmented
meningioma files (`Tr-aug-me_*`), and augmentation applied before splitting puts near-copies
of the same scan on both sides. That is a concrete argument for de-duplicating *before*
splitting rather than after.

Also worth noting: the clean model scores 0.5419 on the leaked slice against 0.8895
elsewhere. Those images are genuinely hard for an honest model, which is exactly why
memorising them pays so well.

### For the rest of the team

- **Laiba** — apply the BatchNorm fix above before fine-tuning EfficientNetB0, and reload the
  checkpoint before scoring. Phase A alone gets you to 0.8551; working fine-tuning should beat it.
- **Mania** — `models/effnet_clean.keras` is the model to run Grad-CAM against. Phase 1 found
  resolution correlates with class (Cramer's V = 0.52), so it is worth confirming the model
  attends to the tumour and not to image dimensions or borders.

### Baseline comparison

The from-scratch CNN reached 0.8060 accuracy / 0.7976 macro-F1, against 0.8551 / 0.8500 for
EfficientNetB0 on the same clean data. Transfer learning is worth about 5 points here — a
real gain, but smaller than the near-perfect figures usually quoted for this dataset, most
of which are measured on uncleaned splits like the leaky one above.